In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    train_df = pd.read_csv('data/train.csv')
    test_df = pd.read_csv('data/test.csv')
    print('✅ Datasets loaded successfully!')
except FileNotFoundError:
    print('❌ Error: File not found. Please double-check the folder path and filenames.')


✅ Datasets loaded successfully!


In [2]:

def gini(p0, p1):
    sum = p0 + p1
    gini_impurity = 1 - (p0/sum)**2 - (p1/sum)**2
    return gini_impurity

def split(df, label, threshold, exact=False):
    if exact:   mask = df[label] == threshold
    else:       mask = df[label] <= threshold

    return df[mask], df[~mask]

def split_quality(df_left, df_right, target_label, id_label):

    sum_left = df_left[id_label].count()
    sum_right = df_right[id_label].count()

    left_counts = df_left[target_label].value_counts()
    right_counts = df_right[target_label].value_counts()

    gini_left = gini(left_counts[0], left_counts[1])
    gini_right = gini(right_counts[0], right_counts[1])

    weighted_gini = sum_left / sum * gini_left + sum_right / sum * gini_right
    return weighted_gini
    

In [3]:
def convert_df(df):
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Age'] = df['Age'].fillna(df['Age'].median())
    return df

train_df = convert_df(train_df)
test_df = convert_df(test_df)

sum = train_df['PassengerId'].count()
print(f"Sum: {sum}")
counts = train_df["Survived"].value_counts()
# print(counts)
print(f"Died: {counts[0]}")
print(f"Survived: {counts[1]}")
gini_impurity = 1 - (counts[0]/sum)**2 - (counts[1]/sum)**2
print(f"Gini impurity: {gini_impurity}")


Sum: 891
Died: 549
Survived: 342
Gini impurity: 0.4730129578614428


In [4]:
print(train_df['Cabin'])

0       NaN
1       C85
2       NaN
3      C123
4       NaN
       ... 
886     NaN
887     B42
888     NaN
889    C148
890     NaN
Name: Cabin, Length: 891, dtype: str


In [5]:
left, right = split(train_df, 'Sex', 0, False)
q = split_quality(left, right, 'Survived', 'PassengerId')
print(q)

0.3333650003885904


In [6]:
left, right = split(train_df, 'Pclass', 2, False)
q = split_quality(left, right, 'Survived', 'PassengerId')
print(q)

0.4238751054331502


In [10]:
left, right = split(train_df, 'Age', 2, False)
q = split_quality(left, right, 'Survived', 'PassengerId')
print(q)

0.46979308657509355


In [7]:
mask = train_df['Sex'].to_numpy() <= 0

df_left = train_df.loc[mask]
df_right = train_df.loc[~mask]
print(df_left)

     PassengerId  Survived  Pclass                            Name  Sex   Age  \
0              1         0       3         Braund, Mr. Owen Harris    0  22.0   
4              5         0       3        Allen, Mr. William Henry    0  35.0   
5              6         0       3                Moran, Mr. James    0  28.0   
6              7         0       1         McCarthy, Mr. Timothy J    0  54.0   
7              8         0       3  Palsson, Master. Gosta Leonard    0   2.0   
..           ...       ...     ...                             ...  ...   ...   
883          884         0       2   Banfield, Mr. Frederick James    0  28.0   
884          885         0       3          Sutehall, Mr. Henry Jr    0  25.0   
886          887         0       2           Montvila, Rev. Juozas    0  27.0   
889          890         1       1           Behr, Mr. Karl Howell    0  26.0   
890          891         0       3             Dooley, Mr. Patrick    0  32.0   

     SibSp  Parch          

In [8]:
train_df["CabinClass"] = train_df["Cabin"].str[0].fillna("X")

survival_by_cabin = train_df.groupby("CabinClass")["Survived"].mean()

# print(survival_by_cabin)

survival_by_cabin = (
    train_df.groupby("CabinClass")["Survived"]
      .agg(["count", "mean"])
      .rename(columns={"count": "Passengers", "mean": "SurvivalRate"})
)

print(survival_by_cabin)

            Passengers  SurvivalRate
CabinClass                          
A                   15      0.466667
B                   47      0.744681
C                   59      0.593220
D                   33      0.757576
E                   32      0.750000
F                   13      0.615385
G                    4      0.500000
T                    1      0.000000
X                  687      0.299854


In [9]:
left, right = split(train_df, 'CabinClass', 'X', True)
q = split_quality(left, right, 'Survived', 'PassengerId')
print(q)

0.4255068883889845
